# Silver | Tratamento: Diretório de UFs
Origem: `workspace.bronze.diretorio_uf_raw` (ingestão 09). Chave de negócio: `id_uf`.
Base de `gold.dim_uf` (id, sigla, nome, região), sem depender de CSV avulso.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.diretorio_uf
USING DELTA AS
SELECT
    CAST(id_uf AS INT) AS id_uf,
    UPPER(TRIM(sigla)) AS sigla_uf,
    TRIM(nome) AS nome_uf,
    TRIM(regiao) AS regiao,
    data_ingestao,
    fonte,
    lote_id
FROM workspace.bronze.diretorio_uf_raw
WHERE id_uf IS NOT NULL AND sigla IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_uf
    ORDER BY data_ingestao DESC
) = 1;

In [0]:
registrar_execucao("SILVER", "diretorio_uf", _m, row_count=contar_tabela("workspace.silver.diretorio_uf"))